# 03 — Generalisation evaluation of the third-party Dhwani checkpoint (pilot)

Scores the **third-party** ONNX model
[`ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model`](https://huggingface.co/ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model)
(MIT, revision `ee01871`, sha256 `d1c232bf…`) on two public benchmarks. We did not train this model and nothing here trains,
tunes or selects anything.

| Dataset | Source (pinned) | Licence | Labels used |
|---|---|---|---|
| ASVspoof 2019 LA **evaluation** partition | Edinburgh DataShare `LA.zip`, DOI 10.7488/ds/2555, 7,640,952,520 B, MD5 `30c98f11…` | ODC-By 1.0 | CM protocol `ASVspoof2019.LA.cm.eval.trl.txt`: bonafide → genuine, spoof → synthetic; attack IDs A07–A19 |
| In-the-Wild | HF `mueller91/In-The-Wild` @ `eee168f`, `release_in_the_wild.zip`, 8,161,489,023 B, sha256 `46665f30…` | **Conflicting**: HF card CC-BY-SA-4.0, project page Apache-2.0 | `meta.csv`: bona-fide → genuine, spoof → synthetic; speaker = public figure |

**What this run is and is not**
- A **pilot subset** (≈ 100 genuine + ≈ 100 synthetic per dataset), selected from metadata only, with a fixed seed, *before* the model runs.
  Pilot numbers are **not** full-dataset benchmark results and must never be reported as such.
- Training-data overlap with Dhwani is **unknown** (see `evaluation/README.md`). Its training code reads an unpublished Kaggle folder.
  ASVspoof's "unseen attacks" (A07–A19 vs. its own train set) are **not** thereby unseen by Dhwani; In-the-Wild's YouTube-sourced bona fide audio
  could overlap with Dhwani's English real audio (14 unnamed YouTube recordings). Neither overlap can be ruled out.
- Fixed analysis plan: preprocessing `train_peak` (provisional primary recipe), our existing windowing (`dhwani_infer.py`, embedded verbatim),
  clip score = mean of window scores, threshold **0.5 (provisional, untuned; not optimised here)**, batch size **1**.
- Scores are uncalibrated softmax outputs, **not probabilities**. Failures are kept and reported; no clip is replaced or dropped silently.
- These clips must never be used for training, threshold tuning, model or preprocessing selection.

**Order:** run cells top to bottom. Runtime: CPU is enough for the pilot (≈ 15–30 min, mostly model time).

In [ ]:
# 1. Settings (fixed BEFORE any result is seen; changing them is a new, separately documented run)
RUN_KIND = "pilot"              # "pilot" | "full" (every clip of both evaluation partitions; see the end of the notebook)
DATASETS = ["asvspoof2019_la_eval", "in_the_wild"]
SELECTION_SEED = 20261025
ASV_N_PER_ATTACK = 8            # 8 x 13 attack IDs (A07-A19) = 104 synthetic (100 rounded up to equal strata)
ASV_N_GENUINE = 104             # matched to the synthetic count
ITW_N_GENUINE = 100
ITW_N_SYNTHETIC = 100
PREPROCESSING = "train_peak"    # provisional primary recipe; do not switch to whichever looks best
ACCESS = "range"                # "range": HTTP range reads of only the needed zip members | "download": whole archives
PROVIDER = "cpu"                # "cpu" (pilot; same as the baseline run) | "cuda" (full eval; CPU-parity-checked first)
PARITY_N = 20                   # with PROVIDER="cuda": first N clips also scored on CPU; abort if they disagree
N_BOOT = 1000                   # bootstrap resamples for AUC/EER/balanced-accuracy CIs (use ~200 for the full run)
USE_DRIVE = False               # True: model cache + run folder in Drive (survives runtime resets, enables resume)
DRIVE_DIR = "/content/drive/MyDrive/vcd_cache"
WORK = "/content/vcd_eval"      # local scratch: archives (download mode), extracted audio
KEEP_AUDIO = True               # pilot: keep extracted clips for listening; full run: False (delete after scoring)
RUN_NAME = None                 # set to an existing run folder name to RESUME it (selection is re-verified)

In [ ]:
# 2. Dependencies (Colab ships most; onnxruntime-gpu only if PROVIDER == "cuda")
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "onnxruntime", "librosa", "soundfile", "huggingface_hub"], check=True)
if PROVIDER == "cuda":
    subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "onnxruntime"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "onnxruntime-gpu"], check=True)

In [ ]:
%%writefile dhwani_infer.py
"""Inference wrapper for the third-party Dhwani ONNX checkpoint.

Model: ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model (MIT),
file best_model.onnx at HF revision MODEL_REVISION. We did NOT train it.

Preprocessing is reproduced from the upstream training notebook
(github.com/ayush2635/Dhwani-Deepfake-Audio-Detection-API @ efd512c,
training/training_notebook.ipynb cell 4), which is the notebook whose cell 14
exports best_model.onnx with a (1, 64000) dummy input. See
docs/UPSTREAM_AUDIT.md for why we do not use the model card's z-score recipe
as the default.

Scores are softmax outputs of an UNCALIBRATED classifier. They are model
scores, not probabilities that a clip is synthetic.
"""

from __future__ import annotations

import hashlib
import json
import time
from dataclasses import dataclass, field, asdict

import numpy as np

# ---- pinned provenance -----------------------------------------------------
MODEL_REPO = "ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model"
MODEL_FILE = "best_model.onnx"
MODEL_REVISION = "ee0187142973e1f02050e47d2a31c55ee0263065"
MODEL_SHA256 = "d1c232bf4d7990526804b375c15804c0ee4e9b566478b908c5a57431491b7342"
MODEL_BYTES = 1262760439

# ---- preprocessing / windowing policy ---------------------------------------
SR = 16000
WIN = 64000            # 4.0 s: training crop length and ONNX export shape
HOP = 32000            # 2.0 s: same hop as the upstream API's sliding window
MIN_DUR_S = 2.0        # brief: 2-10 s clips
MAX_DUR_S = 10.0       # longer clips: analyse the first 10 s, flagged
PEAK_TARGET_DBFS = -3.0

# ---- rejection gates (input validation only; never fed to the model) --------
SILENT_PEAK_DBFS = -50.0   # whole clip quieter than this -> silent
ACTIVE_FRAME_S = 0.025
ACTIVE_REL_DB = 35.0       # frame counts as active if within 35 dB of loudest frame
ACTIVE_ABS_DBFS = -55.0    # ...and above this absolute level
MIN_ACTIVE_S = 1.0

# Class order from upstream dataset.py: labels = [0]*real + [1]*fake.
# Documented in source; must still be confirmed empirically on known clips.
LABELS = {0: "genuine", 1: "synthetic"}
SYNTHETIC_INDEX = 1
DECISION_THRESHOLD = 0.5   # placeholder, NOT tuned on any validation set

# Controlled comparison (docs/UPSTREAM_AUDIT.md):
#   train_peak   - training recipe: peak -3 dBFS, 4 s windows             (A)
#   train_zscore - identical windows, z-score instead of peak norm        (A vs B isolates normalisation)
#   api_replica  - deployed API: first 3 s, z-score, zero-pad to 4 s     (B vs C isolates crop/padding)
PREPROCESSING_MODES = ("train_peak", "train_zscore", "api_replica")


class AudioRejected(Exception):
    """Input cannot be analysed (unreadable, empty, silent, too short)."""


def sha256_file(path: str, chunk: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _dbfs(x: float) -> float:
    return float(20 * np.log10(max(x, 1e-12)))


# ---- loading ------------------------------------------------------------------
def load_audio(path: str):
    """Return (samples [channels, n] float32 at native rate, native_sr).

    Tries soundfile first (wav/flac/ogg), then librosa/audioread (mp3, m4a, opus
    via ffmpeg). Never substitutes zeros on failure.
    """
    import soundfile as sf

    try:
        y, sr = sf.read(path, dtype="float32", always_2d=True)  # (n, ch)
        y = y.T
    except Exception as sf_err:
        try:
            import warnings

            import librosa

            with warnings.catch_warnings():  # expected audioread fallback for m4a/aac
                warnings.simplefilter("ignore", (UserWarning, FutureWarning))
                y, sr = librosa.load(path, sr=None, mono=False)
            y = np.atleast_2d(y).astype(np.float32)
        except Exception as lr_err:
            raise AudioRejected(
                f"unreadable audio (soundfile: {type(sf_err).__name__}; "
                f"librosa: {type(lr_err).__name__}: {lr_err})"
            ) from lr_err
    if y.size == 0 or y.shape[-1] == 0:
        raise AudioRejected("empty audio (0 samples)")
    if not np.all(np.isfinite(y)):
        raise AudioRejected("audio contains NaN/Inf samples")
    return y, int(sr)


def to_mono_16k(y: np.ndarray, sr: int) -> np.ndarray:
    mono = y.mean(axis=0) if y.shape[0] > 1 else y[0]
    if sr != SR:
        import librosa

        mono = librosa.resample(mono, orig_sr=sr, target_sr=SR, res_type="soxr_hq")
    return mono.astype(np.float32)


def signal_stats(x: np.ndarray) -> dict:
    peak = float(np.max(np.abs(x))) if x.size else 0.0
    rms = float(np.sqrt(np.mean(x.astype(np.float64) ** 2))) if x.size else 0.0
    flen = int(ACTIVE_FRAME_S * SR)
    nfr = len(x) // flen
    if nfr:
        fr = x[: nfr * flen].reshape(nfr, flen).astype(np.float64)
        fdb = 10 * np.log10(np.mean(fr**2, axis=1) + 1e-20)
        active = (fdb > fdb.max() - ACTIVE_REL_DB) & (fdb > ACTIVE_ABS_DBFS)
        active_s = float(active.sum() * ACTIVE_FRAME_S)
    else:
        active_s = 0.0
    return {
        "peak_dbfs": round(_dbfs(peak), 2),
        "rms_dbfs": round(_dbfs(rms), 2),
        "active_s": round(active_s, 3),
        "clipping_frac": round(float(np.mean(np.abs(x) >= 0.999)) if x.size else 0.0, 5),
    }


def validate(x16: np.ndarray) -> tuple[np.ndarray, dict]:
    """Apply the 2-10 s policy and silence gates. Returns (analysed audio, info)."""
    dur = len(x16) / SR
    info = {"duration_s": round(dur, 3), "truncated": False}
    if dur < MIN_DUR_S:
        raise AudioRejected(f"too short: {dur:.2f} s < {MIN_DUR_S} s minimum")
    if dur > MAX_DUR_S:
        x16 = x16[: int(MAX_DUR_S * SR)]
        info["truncated"] = True
    info["analysed_duration_s"] = round(len(x16) / SR, 3)
    st = signal_stats(x16)
    info.update(st)
    if st["peak_dbfs"] < SILENT_PEAK_DBFS:
        raise AudioRejected(f"silent: peak {st['peak_dbfs']} dBFS < {SILENT_PEAK_DBFS} dBFS")
    if st["active_s"] < MIN_ACTIVE_S:
        raise AudioRejected(
            f"mostly silent: {st['active_s']:.2f} s active signal < {MIN_ACTIVE_S} s"
        )
    return x16, info


# ---- preprocessing --------------------------------------------------------------
def window_starts(n: int) -> list[int]:
    """4 s windows, 2 s hop, last window aligned to the clip end (full coverage)."""
    if n <= WIN:
        return [0]
    starts = list(range(0, n - WIN + 1, HOP))
    if starts[-1] + WIN < n:
        starts.append(n - WIN)
    return starts


def make_windows(x16: np.ndarray, mode: str) -> tuple[np.ndarray, list[int]]:
    """Return (windows [k, WIN] float32, start sample indices)."""
    if mode in ("train_peak", "train_zscore"):
        if mode == "train_peak":
            # Training notebook: peak-normalise the clip to -3 dBFS, then crop/zero-pad
            # the END to 64000 samples. No mean/variance normalisation.
            peak = float(np.max(np.abs(x16)))
            x = x16 * ((10 ** (PEAK_TARGET_DBFS / 20)) / (peak + 1e-6))
        else:
            # Same windows; z-score over the analysed clip (before padding).
            x = (x16 - x16.mean()) / np.sqrt(x16.var() + 1e-5)
        starts = window_starts(len(x))
        wins = []
        for s in starts:
            w = x[s : s + WIN]
            if len(w) < WIN:
                w = np.pad(w, (0, WIN - len(w)))
            wins.append(w)
        return np.stack(wins).astype(np.float32), starts
    if mode == "api_replica":
        # Deployed upstream API (src/api/inference.py): first 48000 samples,
        # zero-pad to 48000, z-score, then zero-pad to 64000 in predict().
        # Kept only as a comparison; it differs from training.
        w = x16[:48000]
        w = np.pad(w, (0, 48000 - len(w)))
        w = (w - w.mean()) / np.sqrt(w.var() + 1e-5)
        w = np.pad(w, (0, WIN - len(w)))
        return w[None, :].astype(np.float32), [0]
    raise ValueError(f"unknown preprocessing mode {mode!r}; use {PREPROCESSING_MODES}")


def softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


# ---- model ------------------------------------------------------------------------
def load_session(model_path: str, verify_sha: bool = True):
    import onnxruntime as ort

    if verify_sha:
        got = sha256_file(model_path)
        if got != MODEL_SHA256:
            raise RuntimeError(f"model sha256 mismatch: {got} != pinned {MODEL_SHA256}")
    so = ort.SessionOptions()
    sess = ort.InferenceSession(model_path, sess_options=so, providers=["CPUExecutionProvider"])
    return sess


def describe_session(sess) -> dict:
    import onnxruntime as ort

    return {
        "onnxruntime_version": ort.__version__,
        "available_providers": ort.get_available_providers(),
        "session_providers": sess.get_providers(),
        "inputs": [(i.name, i.shape, i.type) for i in sess.get_inputs()],
        "outputs": [(o.name, o.shape, o.type) for o in sess.get_outputs()],
    }


def interface_self_test(sess, seed: int = 0) -> dict:
    """Probe what the checkpoint actually accepts (metadata alone proves nothing)."""
    rng = np.random.default_rng(seed)
    name = sess.get_inputs()[0].name
    out = {}
    base = (rng.standard_normal(WIN) * 0.1).astype(np.float32)
    ref = sess.run(None, {name: base[None]})[0]
    out["len_64000_batch1"] = {"ok": True, "shape": list(ref.shape)}
    for n in (32000, 48000, 96000):
        try:
            r = sess.run(None, {name: base[:n][None] if n <= WIN else np.tile(base, 2)[:n][None]})[0]
            out[f"len_{n}_batch1"] = {"ok": True, "shape": list(r.shape)}
        except Exception as e:
            out[f"len_{n}_batch1"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}
    try:
        b = np.stack([base, base[::-1].copy()])
        r = sess.run(None, {name: b})[0]
        r1 = sess.run(None, {name: base[::-1].copy()[None]})[0]
        # Colab run 20261024T102407Z: batch of 2 returned shape [1, 2] (a row is
        # silently dropped), so "ok" requires one output row per input row.
        out["batch2"] = {
            "ok": r.shape[0] == 2,
            "shape": list(r.shape),
            "max_abs_diff_vs_individual": float(
                max(np.abs(r[0] - ref[0]).max(), np.abs(r[1] - r1[0]).max())
            ) if r.shape[0] == 2 else None,
        }
    except Exception as e:
        out["batch2"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}
    # determinism
    again = sess.run(None, {name: base[None]})[0]
    out["deterministic"] = bool(np.allclose(again, ref))
    return out


@dataclass
class Result:
    file: str
    status: str = "ok"
    reject_reason: str = ""
    file_sha256: str = ""
    native_sr: int | None = None
    channels: int | None = None
    info: dict = field(default_factory=dict)
    preprocessing: str = ""
    window_starts_s: list = field(default_factory=list)
    window_logits: list = field(default_factory=list)
    window_scores: list = field(default_factory=list)
    clip_score_mean: float | None = None
    clip_score_max: float | None = None
    predicted_label: str = ""
    latency_ms_model: float | None = None
    latency_ms_total: float | None = None


def analyse(sess, path: str, mode: str = "train_peak") -> Result:
    """Score an audio file (any format load_audio can decode)."""
    t0 = time.perf_counter()
    res = Result(file=path, preprocessing=mode)
    try:
        res.file_sha256 = sha256_file(path)
        y, sr = load_audio(path)
    except AudioRejected as e:
        res.status, res.reject_reason = "rejected", str(e)
        res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
        return res
    return _score(sess, res, y, sr, t0)


def analyse_waveform(sess, samples: np.ndarray, sr: int, mode: str = "train_peak",
                     label: str = "<waveform>") -> Result:
    """Score in-memory PCM ([n] or [channels, n], float in [-1, 1]).

    Same validation, preprocessing and windowing as analyse(); intended for
    audio windows that never touch disk (e.g. remote-call PCM later).
    """
    t0 = time.perf_counter()
    res = Result(file=label, preprocessing=mode)
    y = np.atleast_2d(np.asarray(samples, dtype=np.float32))
    if y.size == 0:
        res.status, res.reject_reason = "rejected", "empty audio (0 samples)"
    elif not np.all(np.isfinite(y)):
        res.status, res.reject_reason = "rejected", "audio contains NaN/Inf samples"
    if res.status == "rejected":
        res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
        return res
    return _score(sess, res, y, int(sr), t0)


def _score(sess, res: Result, y: np.ndarray, sr: int, t0: float) -> Result:
    try:
        res.native_sr, res.channels = sr, int(y.shape[0])
        x16 = to_mono_16k(y, sr)
        x16, info = validate(x16)
        res.info = info
        wins, starts = make_windows(x16, res.preprocessing)
    except AudioRejected as e:
        res.status, res.reject_reason = "rejected", str(e)
        res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
        return res

    name = sess.get_inputs()[0].name
    logits = []
    tm = time.perf_counter()
    for w in wins:  # batch=1 only: batch>1 drops rows (see interface_self_test)
        logits.append(sess.run(None, {name: w[None]})[0][0])
    res.latency_ms_model = round((time.perf_counter() - tm) * 1000, 1)
    logits = np.asarray(logits, dtype=np.float64)
    scores = softmax(logits)[:, SYNTHETIC_INDEX]
    res.window_starts_s = [round(s / SR, 3) for s in starts]
    res.window_logits = [[round(float(a), 4) for a in row] for row in logits]
    res.window_scores = [round(float(s), 4) for s in scores]
    res.clip_score_mean = round(float(scores.mean()), 4)
    res.clip_score_max = round(float(scores.max()), 4)
    res.predicted_label = LABELS[SYNTHETIC_INDEX] if res.clip_score_mean >= DECISION_THRESHOLD else LABELS[0]
    res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
    return res


def result_row(res: Result, **extra) -> dict:
    d = asdict(res)
    info = d.pop("info")
    for k in ("window_starts_s", "window_logits", "window_scores"):
        d[k] = json.dumps(d[k])
    row = {**extra, **d, **info}
    row.update(
        decision_threshold=DECISION_THRESHOLD,
        label_mapping=json.dumps(LABELS),
        model_repo=MODEL_REPO,
        model_revision=MODEL_REVISION,
        model_sha256=MODEL_SHA256,
    )
    return row

In [ ]:
%%writefile eval_lib.py
"""Helpers for the Dhwani generalisation evaluation (evaluation only, no training).

Standard library only, so the selection, archive access and metric code can be
unit-tested without ML dependencies (evaluation/tests/test_eval_lib.py).
Model execution, preprocessing and windowing are NOT reimplemented here: the
notebook calls src/vcd/dhwani_infer.py (embedded verbatim, read-only).

Labels follow dhwani_infer.LABELS: "genuine" (0) / "synthetic" (1).
Scores are uncalibrated synthetic-class softmax means, not probabilities.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import math
import random
import time
import urllib.error
import urllib.request
from collections import Counter, defaultdict

# ---- pinned dataset sources (verified 2026-10-24, see evaluation/README.md) --------
SOURCES = {
    "asvspoof2019_la_eval": {
        "title": "ASVspoof 2019 Logical Access (LA), evaluation partition",
        "landing_page": "https://datashare.ed.ac.uk/handle/10283/3336",
        "doi": "10.7488/ds/2555",
        "archive": "LA.zip",
        "url": "https://datashare.ed.ac.uk/server/api/core/bitstreams/"
               "a9f87c35-f055-4015-80e2-2fdff0d46269/content",
        "bytes": 7640952520,
        "md5": "30c98f11d8b2bc21f2c257bfd78bb5c5",  # DSpace checkSum (MD5) == HTTP ETag
        "last_modified": "Tue, 04 Jun 2019 09:38:20 GMT",
        "license": "ODC-By 1.0 (archive README.txt, COPYING section)",
        "protocol_suffix": "ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt",
        "audio_dir_suffix": "ASVspoof2019_LA_eval/flac/",
        "audio_ext": ".flac",
        # Literature values, checked against the protocol in the notebook (warn on mismatch).
        "expected_counts": {"genuine": 7355, "synthetic": 63882},
        "expected_attacks": [f"A{i:02d}" for i in range(7, 20)],
    },
    "in_the_wild": {
        "title": "In-the-Wild audio deepfake dataset (Mueller et al., Interspeech 2022)",
        "landing_page": "https://huggingface.co/datasets/mueller91/In-The-Wild",
        "project_page": "https://deepfake-total.com/in_the_wild",
        "hf_repo": "mueller91/In-The-Wild",
        "revision": "eee168f92c367f8c82ff2cf42b6f61e362fd6211",
        "archive": "release_in_the_wild.zip",
        "url": "https://huggingface.co/datasets/mueller91/In-The-Wild/resolve/"
               "eee168f92c367f8c82ff2cf42b6f61e362fd6211/release_in_the_wild.zip",
        "bytes": 8161489023,
        "sha256": "46665f30a6758b45642c659c7e0e80da2ba3d121b064716824a56a010ddf9e1a",  # LFS oid
        "xet_hash": "ec530d2076f973d40a8e388e559888f1b4fea467db8126c811d34dd508ee106f",  # CDN ETag
        "license": "CONFLICT: HF card says CC-BY-SA-4.0; Fraunhofer project page says Apache-2.0",
        "meta_suffix": "meta.csv",
        "expected_counts": {"genuine": 19963, "synthetic": 11816},
    },
}

GENUINE, SYNTHETIC = "genuine", "synthetic"


# ---- remote zip access via HTTP range requests -------------------------------------
class RangeUnsupported(RuntimeError):
    """Server ignored the Range header (would have streamed the whole archive)."""


class HTTPRangeFile(io.RawIOBase):
    """Read-only, seekable view of a remote file using HTTP Range requests.

    zipfile.ZipFile can open it directly, so only the central directory and the
    selected members are transferred. zipfile checks each member's CRC-32 on read.
    `fetch(start, end_inclusive) -> bytes` can be injected for tests.
    """

    def __init__(self, url: str | None = None, size: int | None = None, fetch=None,
                 timeout: float = 60.0, retries: int = 4):
        super().__init__()
        self.url, self.timeout, self.retries = url, timeout, retries
        self._final_url = None
        self._fetch = fetch or self._http_fetch
        self.headers = {}
        if size is None:
            size = self._probe_size()
        self.size, self.pos = int(size), 0
        self.n_requests, self.bytes_fetched = 0, 0

    def _resolve(self):
        req = urllib.request.Request(self.url, headers={"Range": "bytes=0-0"})
        with urllib.request.urlopen(req, timeout=self.timeout) as r:
            if r.status != 206:
                raise RangeUnsupported(f"HTTP {r.status} to a Range request: {self.url}")
            self._final_url = r.geturl()
            self.headers = {k.lower(): v for k, v in r.headers.items()}
            r.read()
        return self.headers

    def _probe_size(self) -> int:
        h = self._resolve()
        return int(h["content-range"].rsplit("/", 1)[1])

    def _http_fetch(self, start: int, end: int) -> bytes:
        last = None
        for attempt in range(self.retries):
            if self._final_url is None:
                self._resolve()
            req = urllib.request.Request(self._final_url, headers={"Range": f"bytes={start}-{end}"})
            try:
                with urllib.request.urlopen(req, timeout=self.timeout) as r:
                    if r.status != 206:
                        raise RangeUnsupported(f"HTTP {r.status} to a Range request")
                    data = r.read()
                if len(data) != end - start + 1:
                    raise IOError(f"short read {len(data)} != {end - start + 1}")
                return data
            except RangeUnsupported:
                raise
            except (urllib.error.URLError, IOError, TimeoutError) as e:
                last = e
                self._final_url = None  # signed redirect URLs expire: resolve again
                time.sleep(2 ** attempt)
        raise IOError(f"range fetch {start}-{end} failed after {self.retries} attempts: {last}")

    # RawIOBase interface
    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            self.pos = offset
        elif whence == io.SEEK_CUR:
            self.pos += offset
        elif whence == io.SEEK_END:
            self.pos = self.size + offset
        else:
            raise ValueError(whence)
        if self.pos < 0:
            raise ValueError("negative seek position")
        return self.pos

    def readinto(self, b):
        n = min(len(b), self.size - self.pos)
        if n <= 0:
            return 0
        data = self._fetch(self.pos, self.pos + n - 1)
        b[: len(data)] = data
        self.pos += len(data)
        self.n_requests += 1
        self.bytes_fetched += len(data)
        return len(data)


def open_remote_zip(url: str, expected_bytes: int | None = None, buffer_size: int = 1 << 18):
    """Return (ZipFile, raw HTTPRangeFile) for a remote archive."""
    import zipfile

    raw = HTTPRangeFile(url)
    if expected_bytes is not None and raw.size != expected_bytes:
        raise IOError(f"remote size {raw.size} != recorded {expected_bytes}")
    zf = zipfile.ZipFile(io.BufferedReader(raw, buffer_size=buffer_size))
    return zf, raw


def find_member(names, suffix: str) -> str:
    hits = [n for n in names if n.endswith(suffix) and not n.split("/")[-1].startswith("._")]
    if len(hits) != 1:
        raise KeyError(f"expected exactly one member ending in {suffix!r}, found {hits[:5]}")
    return hits[0]


def file_digest(path: str, algo: str = "sha256", chunk: int = 1 << 22) -> str:
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


# ---- metadata parsing ------------------------------------------------------------
def parse_asvspoof_cm_protocol(text: str) -> list[dict]:
    """ASVspoof 2019 LA CM protocol: SPEAKER_ID AUDIO_FILE_NAME - SYSTEM_ID KEY."""
    out = []
    for i, line in enumerate(text.splitlines(), 1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"protocol line {i}: expected 5 columns, got {len(parts)}: {line!r}")
        spk, utt, _unused, system, key = parts
        if key not in ("bonafide", "spoof"):
            raise ValueError(f"protocol line {i}: unknown key {key!r}")
        if key == "spoof" and system == "-":
            raise ValueError(f"protocol line {i}: spoof without attack ID")
        out.append({
            "clip_id": utt, "speaker_id": spk,
            "attack_id": system if key == "spoof" else "",
            "label": SYNTHETIC if key == "spoof" else GENUINE,
            "original_label": key,
        })
    return out


ITW_LABELS = {"bona-fide": GENUINE, "bonafide": GENUINE, "spoof": SYNTHETIC}


def parse_itw_meta(text: str) -> list[dict]:
    """In-the-Wild meta.csv (expected columns: file, speaker, label)."""
    rows = list(csv.DictReader(io.StringIO(text)))
    if not rows:
        raise ValueError("meta.csv is empty")
    cols = {c.strip().lower(): c for c in rows[0].keys()}
    missing = {"file", "speaker", "label"} - set(cols)
    if missing:
        raise ValueError(f"meta.csv lacks columns {missing}; has {list(rows[0].keys())}")
    out = []
    for i, r in enumerate(rows, 2):
        lab = r[cols["label"]].strip()
        if lab.lower() not in ITW_LABELS:
            raise ValueError(f"meta.csv line {i}: unknown label {lab!r}")
        fn = r[cols["file"]].strip()
        out.append({
            "clip_id": fn, "speaker_id": r[cols["speaker"]].strip(), "attack_id": "",
            "label": ITW_LABELS[lab.lower()], "original_label": lab,
        })
    return out


def describe_records(records: list[dict]) -> dict:
    by_label = Counter(r["label"] for r in records)
    attacks = Counter(r["attack_id"] for r in records if r["label"] == SYNTHETIC and r["attack_id"])
    spk = defaultdict(Counter)
    for r in records:
        spk[r["label"]][r["speaker_id"]] += 1
    return {
        "n": len(records),
        "by_label": dict(sorted(by_label.items())),
        "by_attack": dict(sorted(attacks.items())),
        "n_speakers_by_label": {k: len(v) for k, v in sorted(spk.items())},
        "n_speakers_total": len({r["speaker_id"] for r in records}),
        "duplicate_clip_ids": len(records) - len({r["clip_id"] for r in records}),
    }


# ---- score-free, reproducible selection ------------------------------------------
FORBIDDEN_SELECTION_KEYS = ("score", "logit", "pred", "prob")


def _rng(seed: int, *stratum) -> random.Random:
    # str seeds are hashed with SHA-512 by random.seed (version 2): stable across
    # runs, machines and PYTHONHASHSEED. One RNG per stratum => order-independent.
    return random.Random(":".join([str(seed), *map(str, stratum)]))


def speaker_round_robin(records: list[dict], n: int, seed: int, *stratum) -> list[dict]:
    """Take n clips spreading over speakers: shuffled speaker order, shuffled clips per
    speaker, then one clip per speaker per round. Uses metadata only."""
    by_spk = defaultdict(list)
    for r in sorted(records, key=lambda r: r["clip_id"]):
        by_spk[r["speaker_id"]].append(r)
    spks = sorted(by_spk)
    _rng(seed, *stratum, "speakers").shuffle(spks)
    for s in spks:
        _rng(seed, *stratum, "clips", s).shuffle(by_spk[s])
    out, rnd = [], 0
    while len(out) < n and any(len(by_spk[s]) > rnd for s in spks):
        for s in spks:
            if len(out) >= n:
                break
            if len(by_spk[s]) > rnd:
                out.append(by_spk[s][rnd])
        rnd += 1
    return out


def select_subset(records: list[dict], dataset: str, seed: int, n_genuine: int,
                  n_synthetic: int | None = None, n_per_attack: int | None = None,
                  full: bool = False) -> tuple[list[dict], dict]:
    """Pilot: score-free stratified sample. full=True: every record (sorted by ID)."""
    for r in records[:1]:
        bad = [k for k in r if any(f in k.lower() for f in FORBIDDEN_SELECTION_KEYS)]
        assert not bad, f"selection input contains detector-derived fields: {bad}"
    gen = [r for r in records if r["label"] == GENUINE]
    syn = [r for r in records if r["label"] == SYNTHETIC]
    if full:
        sel = [dict(r, stratum="all") for r in sorted(records, key=lambda r: r["clip_id"])]
        rule = "full evaluation partition: every protocol/metadata row, sorted by clip_id"
    else:
        sel = [dict(r, stratum="genuine") for r in speaker_round_robin(gen, n_genuine, seed, dataset, GENUINE)]
        if n_per_attack is not None:
            by_atk = defaultdict(list)
            for r in syn:
                by_atk[r["attack_id"]].append(r)
            for atk in sorted(by_atk):
                take = speaker_round_robin(by_atk[atk], n_per_attack, seed, dataset, SYNTHETIC, atk)
                sel += [dict(r, stratum=f"synthetic/{atk}") for r in take]
            rule = (f"genuine: {n_genuine} by speaker round-robin; synthetic: {n_per_attack} per attack ID "
                    f"({len(by_atk)} attacks) by speaker round-robin within attack")
        else:
            sel += [dict(r, stratum="synthetic") for r in speaker_round_robin(syn, n_synthetic, seed, dataset, SYNTHETIC)]
            rule = f"genuine: {n_genuine}, synthetic: {n_synthetic}, each by speaker round-robin"
        rule += (f"; seed {seed}; per-stratum random.Random('{seed}:{dataset}:<label>[:<attack>]:...'); "
                 "input sorted by clip_id; metadata only (no audio, no detector output)")
    for i, r in enumerate(sel):
        r["selection_rank"] = i
        r["dataset"] = dataset
    ids = "\n".join(r["clip_id"] for r in sel)
    meta = {
        "dataset": dataset, "seed": seed, "full": full, "rule": rule,
        "n_selected": len(sel), "selected": describe_records(sel), "available": describe_records(records),
        "clip_id_list_sha256": hashlib.sha256(ids.encode()).hexdigest(),
        "shortfalls": {},
    }
    if not full:
        got = Counter(r["stratum"] for r in sel)
        if got["genuine"] < n_genuine:
            meta["shortfalls"]["genuine"] = f"{got['genuine']} < {n_genuine}"
        for k, v in got.items():
            if k.startswith("synthetic/") and n_per_attack and v < n_per_attack:
                meta["shortfalls"][k] = f"{v} < {n_per_attack}"
    return sel, meta


# ---- metrics (positive class = synthetic; higher score = more synthetic) --------
def _split(labels, scores):
    pos = [s for l, s in zip(labels, scores) if l == 1]
    neg = [s for l, s in zip(labels, scores) if l == 0]
    return pos, neg


def roc_auc(labels, scores):
    """Mann-Whitney U / (n_pos * n_neg), ties counted 1/2. None if a class is empty."""
    pos, neg = _split(labels, scores)
    if not pos or not neg:
        return None
    allv = sorted([(s, 1) for s in pos] + [(s, 0) for s in neg])
    rank_sum_pos, i = 0.0, 0
    while i < len(allv):
        j = i
        while j < len(allv) and allv[j][0] == allv[i][0]:
            j += 1
        avg_rank = (i + 1 + j) / 2.0  # ranks i+1..j
        rank_sum_pos += avg_rank * sum(1 for k in range(i, j) if allv[k][1] == 1)
        i = j
    u = rank_sum_pos - len(pos) * (len(pos) + 1) / 2.0
    return u / (len(pos) * len(neg))


def eer(labels, scores):
    """EER with the ASVspoof 2019 reference convention (compute_eer in the official
    tDCF package): sweep thresholds over all scores, take the point minimising
    |FNR - FPR| and report their mean. Decision: synthetic if score >= t.
    Returns (eer, threshold_at_eer) or (None, None). The EER threshold is
    descriptive only; it is NOT an operating point for this project."""
    pos, neg = _split(labels, scores)
    if not pos or not neg:
        return None, None
    pts = sorted(set(pos) | set(neg)) + [math.inf]  # inf: FNR = 1, FPR = 0
    pos_s, neg_s = sorted(pos), sorted(neg)
    best = None
    ip = ineg = 0
    for t in pts:  # FNR(t) = #pos < t / P ; FPR(t) = #neg >= t / N
        while ip < len(pos_s) and pos_s[ip] < t:
            ip += 1
        while ineg < len(neg_s) and neg_s[ineg] < t:
            ineg += 1
        fnr, fpr = ip / len(pos_s), (len(neg_s) - ineg) / len(neg_s)
        d = abs(fnr - fpr)
        if best is None or d < best[0]:
            best = (d, (fnr + fpr) / 2.0, t)
    return best[1], best[2]


def wilson(k: int, n: int, z: float = 1.959964):
    if n == 0:
        return None, None
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return max(0.0, c - h), min(1.0, c + h)


def at_threshold(labels, scores, thr: float) -> dict:
    tp = sum(1 for l, s in zip(labels, scores) if l == 1 and s >= thr)
    fn = sum(1 for l, s in zip(labels, scores) if l == 1 and s < thr)
    fp = sum(1 for l, s in zip(labels, scores) if l == 0 and s >= thr)
    tn = sum(1 for l, s in zip(labels, scores) if l == 0 and s < thr)
    P, N = tp + fn, fp + tn
    rec = tp / P if P else None
    fpr = fp / N if N else None
    spec = tn / N if N else None
    return {
        "threshold": thr,
        "confusion": {"tp_synthetic_as_synthetic": tp, "fn_synthetic_as_genuine": fn,
                      "fp_genuine_as_synthetic": fp, "tn_genuine_as_genuine": tn},
        "synthetic_recall": rec, "synthetic_recall_wilson95": wilson(tp, P),
        "genuine_fpr": fpr, "genuine_fpr_wilson95": wilson(fp, N),
        "genuine_specificity": spec,
        "balanced_accuracy": (rec + spec) / 2 if rec is not None and spec is not None else None,
        "accuracy": (tp + tn) / (P + N) if P + N else None,
        "n_synthetic": P, "n_genuine": N,
    }


def bootstrap_ci(labels, scores, thr: float, n_boot: int = 1000, seed: int = 0) -> dict:
    """Stratified (within-class) percentile bootstrap, 95 %. Reflects clip sampling
    only; it ignores speaker clustering, so intervals are optimistic."""
    pos, neg = _split(labels, scores)
    if not pos or not neg or n_boot <= 0:
        return {}
    rng = random.Random(f"bootstrap:{seed}")
    acc = defaultdict(list)
    for _ in range(n_boot):
        p = [pos[rng.randrange(len(pos))] for _ in pos]
        q = [neg[rng.randrange(len(neg))] for _ in neg]
        lab, sc = [1] * len(p) + [0] * len(q), p + q
        acc["roc_auc"].append(roc_auc(lab, sc))
        acc["eer"].append(eer(lab, sc)[0])
        acc["balanced_accuracy@thr"].append(at_threshold(lab, sc, thr)["balanced_accuracy"])
    out = {}
    for k, v in acc.items():
        v = sorted(v)
        out[k] = (v[int(0.025 * (len(v) - 1))], v[int(math.ceil(0.975 * (len(v) - 1)))])
    out["n_boot"], out["seed"] = n_boot, seed
    return out


def evaluate(labels, scores, thr: float, n_boot: int = 1000, seed: int = 0) -> dict:
    """labels: 1 = synthetic, 0 = genuine (scored clips only)."""
    e, et = eer(labels, scores)
    return {
        "n_scored": len(labels),
        "threshold_free": {"roc_auc": roc_auc(labels, scores), "eer": e,
                           "eer_threshold_descriptive_only": et},
        "at_threshold": at_threshold(labels, scores, thr),
        "bootstrap95": bootstrap_ci(labels, scores, thr, n_boot, seed),
    }


REJECT_CATEGORIES = (
    ("too short", "too_short"), ("mostly silent", "mostly_silent"), ("silent", "silent"),
    ("unreadable", "unreadable"), ("empty", "empty"), ("NaN", "non_finite"),
)


def reject_category(reason: str) -> str:
    for needle, cat in REJECT_CATEGORIES:
        if reason.startswith(needle) or needle in reason[:40]:
            return cat
    return "other"


def jdump(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

In [ ]:
# 3. Imports, code provenance, runtime report
import os, sys, json, time, hashlib, platform, datetime, importlib, traceback, zipfile, shutil
import numpy as np, pandas as pd
sys.path.insert(0, os.getcwd())
import dhwani_infer as dh, eval_lib as el
importlib.reload(dh); importlib.reload(el)

EXPECTED_CODE_SHA256 = {
    "dhwani_infer.py": "8a7c9c8f4275fb27a0d6ba3dde43aaeedf98974b7057378c37122467ba756077",   # snapshot of src/vcd/dhwani_infer.py at build time
    "eval_lib.py": "922385443e35020b2afb6213a52798e97c90dee606866b93be27c21b9a188639",
}
CODE_SHA256 = {f: hashlib.sha256((open(f, encoding="utf-8").read().rstrip() + "\n").encode()).hexdigest()
               for f in EXPECTED_CODE_SHA256}
assert CODE_SHA256 == EXPECTED_CODE_SHA256, f"embedded code was edited: {CODE_SHA256}"

assert PREPROCESSING == "train_peak", "train_peak is the pre-registered primary recipe for this evaluation"
assert dh.DECISION_THRESHOLD == 0.5, "threshold must stay at the provisional 0.5"
assert dh.LABELS == {0: "genuine", 1: "synthetic"} and dh.SYNTHETIC_INDEX == 1
THRESHOLD = dh.DECISION_THRESHOLD

import onnxruntime as ort, librosa, soundfile, sklearn
ENV = {
    "python": platform.python_version(), "platform": platform.platform(),
    "onnxruntime": ort.__version__, "ort_available_providers": ort.get_available_providers(),
    "numpy": np.__version__, "pandas": pd.__version__, "librosa": librosa.__version__,
    "soundfile": soundfile.__version__, "libsndfile": soundfile.__libsndfile_version__,
    "sklearn": sklearn.__version__, "cpu_count": os.cpu_count(),
    "code_sha256": CODE_SHA256,
}
try:
    import torch
    ENV["torch_cuda"] = torch.cuda.is_available()
    ENV["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except ImportError:
    pass
print(json.dumps(ENV, indent=2))
print("policy:", dict(SR=dh.SR, WIN=dh.WIN, HOP=dh.HOP, MIN_DUR_S=dh.MIN_DUR_S, MAX_DUR_S=dh.MAX_DUR_S,
                      PEAK_TARGET_DBFS=dh.PEAK_TARGET_DBFS, threshold=THRESHOLD, preprocessing=PREPROCESSING))

In [ ]:
# 4. Pinned checkpoint: reuse a cached copy if present, else download once; always verify sha256
from huggingface_hub import hf_hub_download
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

cands = []
if USE_DRIVE:
    cands.append(os.path.join(DRIVE_DIR, dh.MODEL_FILE))
try:
    cands.append(hf_hub_download(dh.MODEL_REPO, dh.MODEL_FILE, revision=dh.MODEL_REVISION, local_files_only=True))
except Exception:
    pass
MODEL_PATH = next((p for p in cands if os.path.exists(p) and os.path.getsize(p) == dh.MODEL_BYTES), None)
MODEL_SOURCE = "cache"
if MODEL_PATH is None:
    MODEL_SOURCE = "downloaded"
    print("No cached copy; downloading", dh.MODEL_REPO, "@", dh.MODEL_REVISION[:7], "(1.26 GB) once ...")
    MODEL_PATH = hf_hub_download(dh.MODEL_REPO, dh.MODEL_FILE, revision=dh.MODEL_REVISION,
                                 **({"local_dir": DRIVE_DIR} if USE_DRIVE else {}))
assert os.path.getsize(MODEL_PATH) == dh.MODEL_BYTES, "unexpected model size"
t0 = time.time()
MODEL_SHA = dh.sha256_file(MODEL_PATH)
assert MODEL_SHA == dh.MODEL_SHA256, f"sha256 mismatch: {MODEL_SHA}"
print(f"{MODEL_SOURCE}: {MODEL_PATH}\nsha256 verified ({time.time()-t0:.1f} s): {MODEL_SHA}")

In [ ]:
# 5. ONNX session(s) + interface probe (batch size 1 only: batch 2 silently drops a row)
if PROVIDER == "cpu":
    sess = dh.load_session(MODEL_PATH, verify_sha=False)        # sha checked above
    cpu_sess = None
elif PROVIDER == "cuda":
    sess = ort.InferenceSession(MODEL_PATH, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
    assert sess.get_providers()[0] == "CUDAExecutionProvider", f"CUDA not active: {sess.get_providers()}"
    cpu_sess = dh.load_session(MODEL_PATH, verify_sha=False)    # parity reference for the first PARITY_N clips
else:
    raise ValueError(PROVIDER)
SESSION = dh.describe_session(sess)
PROBE = dh.interface_self_test(sess)
print(json.dumps({"session": SESSION, "probe": PROBE}, indent=2, default=str))
assert PROBE["len_64000_batch1"]["ok"]
print("batch2 ok?", PROBE["batch2"].get("ok"), "-> regardless, every window is run with batch size 1 (dhwani_infer._score)")

In [ ]:
# 6. Run folder (resumable)
BASE = os.path.join(DRIVE_DIR if USE_DRIVE else WORK, "runs")
utc = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_NAME = RUN_NAME or f"{utc}_{RUN_KIND}_{PREPROCESSING}"
RUN_DIR = os.path.join(BASE, RUN_NAME)
RESUMING = os.path.exists(os.path.join(RUN_DIR, "predictions.jsonl"))
os.makedirs(RUN_DIR, exist_ok=True)
for d in ("archives", "audio"):
    os.makedirs(os.path.join(WORK, d), exist_ok=True)
SETTINGS = dict(RUN_KIND=RUN_KIND, DATASETS=DATASETS, SELECTION_SEED=SELECTION_SEED,
                ASV_N_PER_ATTACK=ASV_N_PER_ATTACK, ASV_N_GENUINE=ASV_N_GENUINE,
                ITW_N_GENUINE=ITW_N_GENUINE, ITW_N_SYNTHETIC=ITW_N_SYNTHETIC,
                PREPROCESSING=PREPROCESSING, THRESHOLD=THRESHOLD, ACCESS=ACCESS, PROVIDER=PROVIDER,
                PARITY_N=PARITY_N, N_BOOT=N_BOOT, KEEP_AUDIO=KEEP_AUDIO, clip_score="mean of window scores",
                batch_size=1)
if not RESUMING:
    el.jdump({"settings": SETTINGS, "env": ENV, "model": {"repo": dh.MODEL_REPO, "revision": dh.MODEL_REVISION,
              "sha256": MODEL_SHA, "path": MODEL_PATH, "source": MODEL_SOURCE},
              "session": SESSION, "probe": PROBE, "started_utc": utc}, os.path.join(RUN_DIR, "run_config.json"))
else:
    prev = json.load(open(os.path.join(RUN_DIR, "run_config.json")))["settings"]
    assert prev == SETTINGS, f"resume with different settings: {prev} vs {SETTINGS}"
print("run dir:", RUN_DIR, "| resuming" if RESUMING else "| new run")

In [ ]:
# 7. Dataset access: open archives, read labels/metadata, check counts. Blockers are recorded, not hidden.
import urllib.parse, urllib.request

def download_archive(ds, src):
    dest = os.path.join(WORK, "archives", src["archive"])
    free = shutil.disk_usage(WORK).free
    if not os.path.exists(dest):
        assert free > src["bytes"] * 1.3, f"not enough disk: {free/1e9:.1f} GB free"
        if ds == "in_the_wild":
            p = hf_hub_download(src["hf_repo"], src["archive"], repo_type="dataset", revision=src["revision"])
            os.symlink(os.path.realpath(p), dest)
        else:
            with urllib.request.urlopen(src["url"], timeout=120) as r, open(dest + ".part", "wb") as f:
                shutil.copyfileobj(r, f, 1 << 22)
            os.rename(dest + ".part", dest)
    assert os.path.getsize(dest) == src["bytes"], "archive size mismatch"
    algo = "sha256" if "sha256" in src else "md5"
    t = time.time(); got = el.file_digest(dest, algo)
    assert got == src[algo], f"{algo} mismatch: {got}"
    return dest, {"access": "full download", "archive_path": dest, f"{algo}_verified": got,
                  "verify_s": round(time.time() - t, 1)}

ARCH, RECORDS, BLOCKERS, ACCESS_INFO, DESC = {}, {}, [], {}, {}
for ds in DATASETS:
    src = el.SOURCES[ds]
    t0 = time.time()
    try:
        if ACCESS == "range":
            zf, raw = el.open_remote_zip(src["url"], src["bytes"])
            etag = (raw.headers.get("etag") or "").strip('"')
            info = {"access": "http range", "host": urllib.parse.urlparse(raw._final_url).netloc,
                    "content_length_total": raw.size, "etag": etag,
                    "last_modified": raw.headers.get("last-modified"),
                    "etag_matches_recorded": etag in (src.get("md5"), src.get("xet_hash")),
                    "integrity_note": "range mode cannot hash the whole archive; size + server ETag are checked "
                                      "and zipfile verifies each extracted member's CRC-32"}
        else:
            path, info = download_archive(ds, src)
            zf, raw = zipfile.ZipFile(path), None
        names = zf.namelist()
        info["n_members"] = len(names)
        if ds == "asvspoof2019_la_eval":
            proto = el.find_member(names, src["protocol_suffix"])
            recs = el.parse_asvspoof_cm_protocol(zf.read(proto).decode("utf-8"))
            audio = {n.rsplit("/", 1)[1][: -len(src["audio_ext"])]: n for n in names
                     if src["audio_dir_suffix"] in n and n.endswith(src["audio_ext"])
                     and "__MACOSX" not in n and not n.rsplit("/", 1)[1].startswith("._")}
            info["metadata_member"] = proto
        else:
            meta = el.find_member(names, src["meta_suffix"])
            recs = el.parse_itw_meta(zf.read(meta).decode("utf-8"))
            prefix = meta[: -len("meta.csv")]
            nameset = set(names)
            audio = {r["clip_id"]: prefix + r["clip_id"] for r in recs if prefix + r["clip_id"] in nameset}
            info["metadata_member"] = meta
        missing = [r["clip_id"] for r in recs if r["clip_id"] not in audio]
        for r in recs:
            r["archive_member"] = audio.get(r["clip_id"], "")
        d = el.describe_records(recs)
        d["n_metadata_rows_without_audio"] = len(missing)
        d["expected_counts"] = src["expected_counts"]
        d["counts_match_expected"] = d["by_label"] == src["expected_counts"]
        if "expected_attacks" in src:
            d["attacks_match_expected"] = sorted(d["by_attack"]) == src["expected_attacks"]
        info["open_s"] = round(time.time() - t0, 1)
        if raw is not None:
            info["requests_so_far"], info["bytes_fetched_so_far"] = raw.n_requests, raw.bytes_fetched
        ARCH[ds], RECORDS[ds], ACCESS_INFO[ds], DESC[ds] = zf, recs, info, d
        print(f"== {ds}\n", json.dumps({"access": info, "metadata": d}, indent=1))
        if not d["counts_match_expected"]:
            print(f"WARNING {ds}: label counts differ from literature values {src['expected_counts']}")
        if missing:
            print(f"WARNING {ds}: {len(missing)} metadata rows have no audio member (kept as fetch failures if selected)")
    except Exception as e:
        BLOCKERS.append({"dataset": ds, "stage": "access", "error": f"{type(e).__name__}: {e}",
                         "traceback": traceback.format_exc()[-2000:]})
        print(f"BLOCKER {ds}: {type(e).__name__}: {e}")
el.jdump({"sources": {k: el.SOURCES[k] for k in DATASETS}, "access": ACCESS_INFO, "metadata": DESC,
          "blockers": BLOCKERS}, os.path.join(RUN_DIR, "sources_and_access.json"))
print("blockers:", BLOCKERS or "none")

In [ ]:
# 8. Score-free selection -> manifests written BEFORE any audio is scored
MANIFEST = []
for ds, recs in RECORDS.items():
    full = RUN_KIND == "full"
    if ds == "asvspoof2019_la_eval":
        sel, smeta = el.select_subset(recs, ds, SELECTION_SEED, ASV_N_GENUINE, n_per_attack=ASV_N_PER_ATTACK, full=full)
    else:
        sel, smeta = el.select_subset(recs, ds, SELECTION_SEED, ITW_N_GENUINE, n_synthetic=ITW_N_SYNTHETIC, full=full)
    smeta.update(source=el.SOURCES[ds], python=platform.python_version())
    sp = os.path.join(RUN_DIR, f"selection_{ds}.json")
    if os.path.exists(sp):   # resume: selection must be byte-for-byte the same list
        prev = json.load(open(sp))
        assert prev["clip_id_list_sha256"] == smeta["clip_id_list_sha256"], "selection changed on resume"
    else:
        el.jdump(smeta, sp)
        pd.DataFrame(sel).to_csv(os.path.join(RUN_DIR, f"manifest_{ds}.csv"), index=False)
    MANIFEST += sel
    print(f"== {ds}: {smeta['n_selected']} selected | {smeta['rule']}")
    print("   by label:", smeta["selected"]["by_label"], "| by attack:", smeta["selected"]["by_attack"] or "-",
          "| speakers:", smeta["selected"]["n_speakers_by_label"])
    print("   clip-ID list sha256:", smeta["clip_id_list_sha256"], "| shortfalls:", smeta["shortfalls"] or "none")
assert MANIFEST, f"nothing selected; blockers: {BLOCKERS}"
pd.DataFrame(MANIFEST)[["dataset", "selection_rank", "stratum", "clip_id", "label", "speaker_id", "attack_id"]].head(12)

In [ ]:
# 9. Fetch + score every selected clip (batch size 1, train_peak, mean of windows). Appends to predictions.jsonl (resumable).
PRED_PATH = os.path.join(RUN_DIR, "predictions.jsonl")
done = set()
if os.path.exists(PRED_PATH):
    for line in open(PRED_PATH, encoding="utf-8"):
        r = json.loads(line); done.add((r["dataset"], r["clip_id"]))
todo = [m for m in MANIFEST if (m["dataset"], m["clip_id"]) not in done]
print(f"{len(MANIFEST)} selected, {len(done)} already scored, {len(todo)} to go")

parity = []
t_start = time.time()
with open(PRED_PATH, "a", encoding="utf-8") as out:
    for i, m in enumerate(todo):
        ds = m["dataset"]
        row = {k: m[k] for k in ("dataset", "selection_rank", "stratum", "clip_id", "label", "original_label",
                                 "speaker_id", "attack_id", "archive_member")}
        local = os.path.join(WORK, "audio", ds, os.path.basename(m["archive_member"] or m["clip_id"]))
        tf = time.perf_counter()
        try:
            if not m["archive_member"]:
                raise KeyError("no audio member for this metadata row")
            data = ARCH[ds].read(m["archive_member"])          # zipfile verifies CRC-32
            os.makedirs(os.path.dirname(local), exist_ok=True)
            with open(local, "wb") as f:
                f.write(data)                                   # byte copy, no re-encoding
            row.update(fetch_status="ok", fetch_error="", archive_bytes=len(data))
            try:   # container duration, recorded even for clips the policy later rejects
                inf = soundfile.info(local)
                row.update(container_duration_s=round(inf.duration, 3), container_sr=inf.samplerate,
                           container_format=f"{inf.format}/{inf.subtype}")
            except Exception as e:
                row.update(container_duration_s=None, container_sr=None, container_format=f"unreadable: {type(e).__name__}")
        except Exception as e:
            row.update(fetch_status="failed", fetch_error=f"{type(e).__name__}: {e}", archive_bytes=None)
        row["fetch_ms"] = round((time.perf_counter() - tf) * 1000, 1)
        if row["fetch_status"] == "ok":
            r = dh.analyse(sess, local, mode=PREPROCESSING)
            row.update(dh.result_row(r, provider=sess.get_providers()[0]))
            if cpu_sess is not None and len(parity) < PARITY_N and r.status == "ok":
                rc = dh.analyse(cpu_sess, local, mode=PREPROCESSING)
                parity.append(abs(rc.clip_score_mean - r.clip_score_mean))
                if len(parity) == PARITY_N:
                    print("CUDA vs CPU max |clip score diff| over", PARITY_N, "clips:", max(parity))
                    assert max(parity) <= 2e-3, "CUDA scores disagree with CPU; use PROVIDER='cpu'"
            if not KEEP_AUDIO:
                os.remove(local)
        else:
            row.update(status="fetch_failed", reject_reason=row["fetch_error"])
        row["reject_category"] = (el.reject_category(row.get("reject_reason", "")) if row.get("status") == "rejected"
                                  else ("fetch_failed" if row.get("status") == "fetch_failed" else ""))
        out.write(json.dumps(row, default=str) + "\n"); out.flush()
        if (i + 1) % 25 == 0 or i + 1 == len(todo):
            el_s = time.time() - t_start
            print(f"{i+1}/{len(todo)}  {el_s/60:.1f} min elapsed, ETA {el_s/(i+1)*(len(todo)-i-1)/60:.1f} min")
if parity:
    el.jdump({"n": len(parity), "max_abs_diff": max(parity)}, os.path.join(RUN_DIR, "cuda_cpu_parity.json"))

In [ ]:
# 10. Load saved predictions (READ-ONLY: nothing is re-scored or rewritten); dataset status, coverage, exclusions
import os, json
import numpy as np, pandas as pd

PLANNED_N = {"asvspoof2019_la_eval": ASV_N_GENUINE + 13 * ASV_N_PER_ATTACK,   # 104 + 8 x 13 = 208
             "in_the_wild": ITW_N_GENUINE + ITW_N_SYNTHETIC}                  # 200
PRED_PATH = os.path.join(RUN_DIR, "predictions.jsonl")
_raw = [json.loads(l) for l in open(PRED_PATH, encoding="utf-8") if l.strip()]
_acc_path = os.path.join(RUN_DIR, "sources_and_access.json")
_acc = json.load(open(_acc_path, encoding="utf-8")) if os.path.exists(_acc_path) else {}
_blockers = _acc.get("blockers", []) + [b for b in globals().get("BLOCKERS", []) if b not in _acc.get("blockers", [])]

# --- which datasets are actually present, and why -------------------------------------------
DATASET_STATUS = {}
for ds in DATASETS:
    man = os.path.join(RUN_DIR, f"manifest_{ds}.csv")
    n_man = len(pd.read_csv(man)) if os.path.exists(man) else 0
    n_pred = sum(1 for r in _raw if r.get("dataset") == ds)
    errs = sorted({b["error"] for b in _blockers if b.get("dataset") == ds})
    if n_pred == 0:
        status = "NOT TESTED"
        why = ("access blocker: " + " | ".join(errs)) if errs else (
            "no manifest written (dataset never reached selection)" if not n_man else "selected but never scored")
    elif n_pred < n_man:
        status, why = "INCOMPLETE", f"{n_pred}/{n_man} manifest clips have predictions"
    else:
        status, why = "scored", ""
    DATASET_STATUS[ds] = {"status": status, "reason": why, "planned_clips": PLANNED_N.get(ds),
                          "manifest_clips": n_man, "prediction_rows": n_pred,
                          "metadata": _acc.get("metadata", {}).get(ds), "access": _acc.get("access", {}).get(ds),
                          "blockers": errs}
print(f"predictions.jsonl: {len(_raw)} rows (read only) | planned {sum(PLANNED_N.values())} | "
      f"in-memory MANIFEST: {len(MANIFEST)}")
for ds, s in DATASET_STATUS.items():
    print(f"  {ds:22s} {s['status']:10s} planned {s['planned_clips']}, manifest {s['manifest_clips']}, "
          f"predictions {s['prediction_rows']}  {s['reason']}")
TESTED = [ds for ds, s in DATASET_STATUS.items() if s["prediction_rows"] > 0]
assert TESTED, f"no dataset has predictions: {DATASET_STATUS}"
NOT_TESTED = [ds for ds in DATASETS if ds not in TESTED]
el.jdump(DATASET_STATUS, os.path.join(RUN_DIR, "dataset_status.json"))   # new file; predictions untouched

# --- predictions of present datasets only ----------------------------------------------------
P = pd.DataFrame([r for r in _raw if r.get("dataset") in TESTED])
P = P.drop_duplicates(["dataset", "clip_id"], keep="last").sort_values(["dataset", "selection_rank"]).reset_index(drop=True)
for ds in TESTED:
    assert (P.dataset == ds).sum() == DATASET_STATUS[ds]["manifest_clips"], f"{ds}: predictions != manifest"
for c in ("attack_id", "reject_category", "reject_reason", "window_scores", "truncated",
          "duration_s", "container_duration_s"):
    if c not in P:
        P[c] = np.nan
P["attack_id"] = P["attack_id"].fillna("").astype(str)
P["reject_category"] = P["reject_category"].fillna("").astype(str)
P["n_windows"] = P["window_scores"].map(lambda s: len(json.loads(s)) if isinstance(s, str) else 0)
P["attack_or_label"] = np.where(P.label == "synthetic", P.attack_id.where(P.attack_id != "", "synthetic"), "genuine")
P["_truncated"] = P["truncated"].map(lambda v: v is True or str(v).lower() == "true")

COV_COLS = ["selected", "fetch_failed", "rejected", "scored", "coverage", "truncated_to_10s", "single_padded_window(<=4s)"]
def _cov(g):
    ok = g.status == "ok"
    return {"selected": len(g), "fetch_failed": int((g.status == "fetch_failed").sum()),
            "rejected": int((g.status == "rejected").sum()), "scored": int(ok.sum()),
            "coverage": round(float(ok.mean()), 4) if len(g) else None,
            "truncated_to_10s": int(g._truncated.sum()),
            "single_padded_window(<=4s)": int((ok & (g.n_windows == 1)).sum())}

COV = pd.DataFrame([{"dataset": ds, "label": lab, **_cov(g)} for (ds, lab), g in P.groupby(["dataset", "label"])],
                   columns=["dataset", "label"] + COV_COLS)
_asv = P[P.dataset == "asvspoof2019_la_eval"]
COV_ATTACK = pd.DataFrame([{"attack_or_label": k, **_cov(g)} for k, g in _asv.groupby("attack_or_label")],
                          columns=["attack_or_label"] + COV_COLS)
_bad = P[P.status != "ok"]
REJ = pd.DataFrame([{"dataset": d, "label": l, "reject_category": c, "n": len(g)}
                    for (d, l, c), g in _bad.groupby(["dataset", "label", "reject_category"])],
                   columns=["dataset", "label", "reject_category", "n"])
EXCLUDED = _bad.reindex(columns=["dataset", "clip_id", "label", "attack_id", "speaker_id", "status",
                                 "reject_category", "reject_reason", "container_duration_s"])
_okp = P[P.status == "ok"]
DUR = (_okp.groupby(["dataset", "label"])["duration_s"].describe(percentiles=[.05, .5, .95]).round(2)
       if len(_okp) else pd.DataFrame())

print("\nCoverage by dataset x label:\n", COV.to_string(index=False))
print("\nASVspoof coverage by attack:\n", COV_ATTACK.to_string(index=False) if len(COV_ATTACK)
      else "  none: ASVspoof 2019 LA was NOT TESTED in this run")
print("\nRejection / fetch-failure reasons:\n", REJ.to_string(index=False) if len(REJ) else "  none")
print("\nDurations of scored clips (s):\n", DUR.to_string() if len(DUR) else "  none")
print("\nPolicy: < 2.0 s rejected (not replaced); > 10.0 s -> first 10 s only; <= 4 s -> one zero-padded window; "
      "> 4 s -> 4 s windows, 2 s hop, last aligned to end. Metrics use scored clips only.")
for ds in NOT_TESTED:
    print(f"\n*** {ds}: NOT TESTED in this run ({DATASET_STATUS[ds]['reason']}). "
          "No metric below refers to it; do not report it as evaluated. ***")

In [ ]:
# 11. Metrics by dataset (threshold-free vs. at the provisional 0.5) + per attack; sklearn cross-check
from sklearn.metrics import roc_auc_score, roc_curve
OK = P[P.status == "ok"].copy()
OK["y"] = (OK.label == "synthetic").astype(int)
OK["score"] = OK.clip_score_mean.astype(float)
METRICS, PER_ATTACK, XCHECK = {}, [], {}
for ds, g in OK.groupby("dataset"):
    y, s = g.y.tolist(), g.score.tolist()
    m = el.evaluate(y, s, THRESHOLD, n_boot=N_BOOT, seed=SELECTION_SEED)
    m["score_quantiles"] = g.groupby("label").score.describe(percentiles=[.05, .25, .5, .75, .95]).round(4).to_dict("index")
    METRICS[ds] = m
    if len(set(y)) == 2:
        fpr, tpr, _ = roc_curve(y, s, drop_intermediate=False); fnr = 1 - tpr
        k = int(np.nanargmin(np.abs(fnr - fpr)))
        XCHECK[ds] = {"auc_ours": m["threshold_free"]["roc_auc"], "auc_sklearn": float(roc_auc_score(y, s)),
                      "eer_ours": m["threshold_free"]["eer"], "eer_sklearn_convention": float((fnr[k] + fpr[k]) / 2)}
        assert abs(XCHECK[ds]["auc_ours"] - XCHECK[ds]["auc_sklearn"]) < 1e-9, XCHECK[ds]
        if abs(XCHECK[ds]["eer_ours"] - XCHECK[ds]["eer_sklearn_convention"]) > 1e-6:
            print("NOTE: EER tie-break differs from sklearn sweep:", XCHECK[ds])
    gen = g[g.y == 0]
    for atk, ga in g[(g.y == 1) & (g.attack_id != "")].groupby("attack_id"):
        yy, ss = [1] * len(ga) + [0] * len(gen), ga.score.tolist() + gen.score.tolist()
        mt = el.evaluate(yy, ss, THRESHOLD, n_boot=0)
        PER_ATTACK.append({"dataset": ds, "attack_id": atk, "n_synthetic_scored": len(ga), "n_genuine_scored_pooled": len(gen),
                           "roc_auc_vs_all_genuine": mt["threshold_free"]["roc_auc"], "eer_vs_all_genuine": mt["threshold_free"]["eer"],
                           "synthetic_recall@0.5": mt["at_threshold"]["synthetic_recall"],
                           "recall_wilson95": mt["at_threshold"]["synthetic_recall_wilson95"],
                           "median_score": float(ga.score.median())})
PER_ATTACK = pd.DataFrame(PER_ATTACK)

def fmt(x, nd=4):
    if isinstance(x, (tuple, list)) and None in x:
        return "n/a"
    return "n/a" if x is None else (f"[{x[0]:.{nd}f}, {x[1]:.{nd}f}]" if isinstance(x, (tuple, list)) else f"{x:.{nd}f}")
for ds, m in METRICS.items():
    tf, at, bs = m["threshold_free"], m["at_threshold"], m["bootstrap95"]
    c = at["confusion"]
    print(f"===== {ds}  [{RUN_KIND.upper()} SUBSET, n_scored={m['n_scored']}; not a full-dataset benchmark]" if RUN_KIND == "pilot"
          else f"===== {ds}  [FULL evaluation partition, n_scored={m['n_scored']}]")
    print(f"Threshold-free: ROC-AUC {fmt(tf['roc_auc'])} (95% boot {fmt(bs.get('roc_auc'))}) | "
          f"EER {fmt(tf['eer'])} (95% boot {fmt(bs.get('eer'))}); EER threshold {fmt(tf['eer_threshold_descriptive_only'])} is descriptive only")
    print(f"At threshold {THRESHOLD} (provisional, untuned):")
    print(f"   confusion  pred_synthetic  pred_genuine\n   synthetic  {c['tp_synthetic_as_synthetic']:>14}  {c['fn_synthetic_as_genuine']:>12}"
          f"\n   genuine    {c['fp_genuine_as_synthetic']:>14}  {c['tn_genuine_as_genuine']:>12}")
    print(f"   balanced accuracy {fmt(at['balanced_accuracy'])} (95% boot {fmt(bs.get('balanced_accuracy@thr'))})")
    print(f"   synthetic recall  {fmt(at['synthetic_recall'])} (Wilson {fmt(at['synthetic_recall_wilson95'])})")
    print(f"   genuine FPR       {fmt(at['genuine_fpr'])} (Wilson {fmt(at['genuine_fpr_wilson95'])})")
    print("   CIs: clip-level resampling only; speaker clustering ignored -> optimistic.\n")
if len(PER_ATTACK):
    print("Per attack (ASVspoof; each attack vs ALL scored genuine pilot clips; n per attack is tiny):")
    print(PER_ATTACK.round(4).to_string(index=False))
print("\nsklearn cross-check:", json.dumps(XCHECK, indent=1))

In [ ]:
# 12. Runtime
RUNTIME = {}
for ds, g in P.groupby("dataset"):
    ok = g[g.status == "ok"]
    win = ok.latency_ms_model / ok.n_windows.clip(lower=1)
    RUNTIME[ds] = {
        "clips_scored": int(len(ok)), "windows_scored": int(ok.n_windows.sum()),
        "model_ms_per_clip_median": float(ok.latency_ms_model.median()) if len(ok) else None,
        "model_ms_per_window_median": float(win.median()) if len(ok) else None,
        "total_ms_per_clip_median": float(ok.latency_ms_total.median()) if len(ok) else None,
        "fetch_ms_per_clip_median": float(g.fetch_ms.median()),
        "sum_total_min": round(float(g.latency_ms_total.fillna(0).sum() + g.fetch_ms.sum()) / 60000, 2),
        "provider": sess.get_providers()[0],
    }
    exp_n = sum(el.SOURCES[ds]["expected_counts"].values())
    if len(ok):
        RUNTIME[ds]["rough_full_partition_hours_same_hardware"] = round(
            exp_n * float((ok.latency_ms_total.mean() + g.fetch_ms.mean())) / 3.6e6, 1)
print(json.dumps(RUNTIME, indent=1))

In [ ]:
# 13. Save everything (+ zip for download). Per-file predictions include every selected clip, failures too.
P.to_csv(os.path.join(RUN_DIR, "predictions.csv"), index=False)
COV.to_csv(os.path.join(RUN_DIR, "coverage_by_dataset_label.csv"), index=False)
COV_ATTACK.to_csv(os.path.join(RUN_DIR, "coverage_by_attack.csv"), index=False)
EXCLUDED.to_csv(os.path.join(RUN_DIR, "excluded_clips.csv"), index=False)
PER_ATTACK.to_csv(os.path.join(RUN_DIR, "metrics_by_attack.csv"), index=False)
SUMMARY = {
    "run_name": RUN_NAME, "run_kind": RUN_KIND,
    "scope_warning": ("PILOT SUBSET ONLY - not a full-dataset benchmark result" if RUN_KIND == "pilot"
                      else "full evaluation partitions"),
    "overlap_with_dhwani_training": "unknown (training data not published; see evaluation/README.md)",
    "datasets_tested": TESTED, "datasets_NOT_tested": NOT_TESTED, "dataset_status": DATASET_STATUS,
    "settings": SETTINGS, "env": ENV, "model": {"repo": dh.MODEL_REPO, "revision": dh.MODEL_REVISION, "sha256": MODEL_SHA},
    "access": ACCESS_INFO, "blockers": BLOCKERS, "metadata": DESC,
    "coverage": COV.to_dict("records"), "rejections": REJ.to_dict("records"),
    "metrics_by_dataset": METRICS, "sklearn_crosscheck": XCHECK, "runtime": RUNTIME,
    "finished_utc": datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
}
el.jdump(SUMMARY, os.path.join(RUN_DIR, "metrics.json"))
zip_path = shutil.make_archive(os.path.join(WORK, RUN_NAME), "zip", RUN_DIR)
print("saved:", sorted(os.listdir(RUN_DIR)), "\nzip:", zip_path, f"({os.path.getsize(zip_path)/1e6:.2f} MB, no audio inside)")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass

## Report back
1. The downloaded `<run>.zip` (predictions, manifests, selection JSONs, coverage, metrics, access/provenance). It contains **no audio**.
2. The executed notebook (*File → Download → .ipynb*).
3. Any blocker printed by step 7, verbatim.

## Path to the full evaluation (after the pilot)
The pilot answers "does the pipeline work and roughly where does the model sit"; it is **not** a benchmark. For the full run:
1. Keep every setting except: `RUN_KIND = "full"`, `ACCESS = "download"` (verifies the whole-archive MD5 / sha256),
   `KEEP_AUDIO = False`, `N_BOOT = 200`, `USE_DRIVE = True` (the run folder then survives resets; set `RUN_NAME` to resume).
2. Full scope: ASVspoof 2019 LA eval = every protocol row (literature: 7,355 bona fide + 63,882 spoof); In-the-Wild = every `meta.csv` row
   (literature: 19,963 + 11,816). Step 12 prints a CPU time projection from the pilot; at ≈ 2 s per clip it is **tens of hours**, so
   use `PROVIDER = "cuda"` on a GPU runtime (the first `PARITY_N` clips are also scored on CPU and the run aborts if they disagree by > 0.002).
   Resume across sessions with the same `RUN_NAME`.
3. Disk: ≈ 16 GB of archives on local `/content` (not Drive) plus one extracted clip at a time.
4. Report the full run as a new, separate result; never merge pilot and full numbers. Keep `train_peak` and 0.5 unless a decision
   made on **separate validation data** (not these clips) is recorded in `docs/PROGRESS.md` first.